In [1]:
!pip install uv
!uv venv /content/sgl_env --python 3.11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 47.3 MB/s eta 0:00:00
Using CPython 3.11.16
Creating virtual environment at: sgl_env
Activate with: source sgl_env/bin/activate


In [2]:
!uv pip install --python /content/sgl_env/bin/python "sglang[all]"

Using Python 3.11.16 environment at: sgl_env
Resolved 244 packages in 4.55s
Prepared 243 packages in 1m 21s
Installed 244 packages in 1.60s
 + addict==2.4.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + airportsdata==20260905
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==1.7.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.15.1
 + apache-tvm-ffi==0.1.11
 + asttokens==3.0.2
 + attrs==26.1.0
 + av==16.1.0
 + blobfile==3.0.0
 + build==1.6.1
 + cache-dit==1.3.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + compressed-tensors==0.18.0
 + cryptography==50.0.1
 + cuda-bindings==13.4.2
 + cuda-core==1.2.0
 + cuda-pathfinder==1.8.2
 + cuda-python==13.4.1
 + cuda-tile==1.6.0rc5
 + cuda-toolkit==13.0.3.0
 + cython==3.3.0
 + datasets==5.0.1
 + decorator==5.3.1
 + diffusers==0.37.0
 + dill==0.4.1
 + diskcache==5.6.3
 + distro==1.9.0
 + docstring-parser==0.18.0
 + easydict==1.13
 + einops==0.8.2
 + exe

In [11]:
!apt-get install -y ninja-build
!uv pip install --python /content/sgl_env/bin/python ninja

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  ninja-build
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 129 kB of archives.
After this operation, 364 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 ninja-build amd64 1.11.1-2 [129 kB]
Fetched 129 kB in 1s (116 kB/s)
Selecting previously unselected package ninja-build.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.11.1-2_amd64.deb ...
Unpacking ninja-build (1.11.1-2) ...
Setting up ninja-build (1.11.1-2) ...
Processing triggers for man-db (2.12.0-4build2) ...
Using Python 3.11.16 environment at: sgl_env
Checked 1 package in 155ms


In [13]:
%%writefile test_sglang.py
import sglang as sgl
import torch

def main():
    print(f">>> PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
    print(">>> Dang khoi tao SGLang Engine voi EAGLE (Eager Mode)...")

    llm = sgl.Engine(
        model_path="Qwen/Qwen2.5-7B-Instruct-AWQ",
        speculative_draft_model_path="leptonai/EAGLE-Qwen2.5-7B-Instruct",
        speculative_algorithm="EAGLE",
        speculative_num_steps=3,
        speculative_eagle_topk=1,
        speculative_num_draft_tokens=4,
        mem_fraction_static=0.75,
        context_length=2048,
        trust_remote_code=True,
        attention_backend="triton",
        disable_cuda_graph=True       # TẮT CUDA GRAPH ĐỂ TRÁNH LỖI KERNEL TRÊN GPU T4
    )

    prompt = "Toi la sinh vien PTIT dang lam do an tot nghiep. Hay cho toi 3 loi khuyen ngan gon."

    sampling_params = {
        "temperature": 0.7,
        "max_new_tokens": 128
    }

    print(">>> Dang sinh van ban...")
    output = llm.generate(prompt, sampling_params)
    print("\n--- KET QUA ---")
    print(output["text"])

    llm.shutdown()

if __name__ == "__main__":
    main()


Overwriting test_sglang.py


In [15]:
# Vá file cuda_graph_buffer_registry.py trên đĩa
patch_file = "/content/sgl_env/lib/python3.11/site-packages/sglang/srt/model_executor/cuda_graph_buffer_registry.py"

with open(patch_file, "r") as f:
    code = f.read()

# Thay thế lệnh foreach_copy lỗi bằng vòng lặp copy_ chuẩn
old_code = "torch._foreach_copy_(group_dsts, group_srcs)"
new_code = "for d, s in zip(group_dsts, group_srcs): d.copy_(s)"

if old_code in code:
    code = code.replace(old_code, new_code)
    with open(patch_file, "w") as f:
        f.write(code)
    print(">>> ĐÃ VÁ THÀNH CÔNG FILE cuda_graph_buffer_registry.py!")
else:
    print(">>> Không tìm thấy dòng mã cần vá hoặc file đã được vá trước đó.")

>>> ĐÃ VÁ THÀNH CÔNG FILE cuda_graph_buffer_registry.py!


In [16]:
!/content/sgl_env/bin/python test_sglang.py

>>> PyTorch: 2.13.0+cu130 | CUDA: True
>>> Dang khoi tao SGLang Engine voi EAGLE (Eager Mode)...
[2026-09-19 02:58:55] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


>>> Dang sinh van ban...
[2026-09-19 02:59:56] Scheduler hit an exception: Traceback (most recent call last):
  File "/content/sgl_env/lib/python3.11/site-packages/sglang/srt/managers/scheduler.py", line 5857, in run_scheduler_process
    scheduler.run_event_loop()
  File "/content/sgl_env/lib/python3.11/site-packages/sglang/srt/managers/scheduler.py", line 1893, in run_event_loop
    dispatch_event_loop(self)
  File "/content/sgl_env/lib/python3.11/site-packages/sglang/srt/managers/scheduler.py", line 5703, in dispatch_event_loop
    scheduler.event_loop_overlap()
  File "/content/sgl_env/lib/python3.11/site-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^

In [ ]:
import time
from openai import OpenAI

client = OpenAI(base_url="http://localhost:30000/v1", api_key="EMPTY")

prompt = "Viết một bài văn ngắn khoảng 150 từ ca ngợi vẻ đẹp của Vịnh Hạ Long bằng tiếng Việt."

start_time = time.time()
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.0,
    max_tokens=256
)
elapsed = time.time() - start_time

text = response.choices[0].message.content
total_tokens = response.usage.completion_tokens

print(f"\n[KẾT QUẢ SGLANG + EAGLE-2]")
print(f"- Số token sinh ra: {total_tokens}")
print(f"- Thời gian: {elapsed:.2f} giây")
print(f"- Tốc độ: {total_tokens / elapsed:.2f} tok/s")
print(f"\n[Nội dung]:\n{text}")